# 07 · Student Candidate — MobileNetV2 (pretrained ImageNet)

Standard MobileNetV2 fine-tuned from ImageNet weights.
~3.4M parameters.

## Role in the 2×2 comparison

| | Scratch | Pretrained |
|---|---|---|
| **MobileNetV2** | notebook 06 | **← this notebook** |
| **MobileNetV3** | notebook 08 | notebook 09 |

**Claim this notebook helps answer:**
- *Initialization effect on V2:* Does pretraining improve MobileNetV2 on VWW?
  Compare this notebook's result against notebook 06 (same architecture, random init).

**Training protocol:**
- Phase 1 (epochs 1–9):  backbone frozen, classifier head only, lr=3e-4
- Phase 2 (epochs 10–25): full unfreeze at lr=1e-4


In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/thesis/utils/")


In [ ]:
# ── Standard imports ────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import (
    MobileNetV2_Scratch, MobileNetV2_Pretrained,
    MobileNetV3_Scratch, MobileNetV3_Pretrained,
    count_params, model_size_mb, STUDENT_REGISTRY,
)
from utils.train import (
    setup_device, set_seed, evaluate,
    train_multi_seed, plot_history,
)

device = setup_device(seed=41)


In [ ]:
# ── Dataset setup ───────────────────────────────────────────────────
prepare_dataset()


## Standardized hyperparameters

All four student models use **identical training conditions** to ensure the
comparison is controlled — only architecture and initialization vary.

| Parameter | Scratch models | Pretrained models |
|-----------|---------------|-------------------|
| Batch size | 64 | 64 |
| Optimizer | Adam | Adam |
| Weight decay | 1e-4 | 1e-4 |
| Label smoothing | 0.1 | 0.1 |
| Augmentation | standard | standard |
| Scheduler | CosineAnnealingLR | CosineAnnealingLR |
| Patience | 10 | 10 |
| Seeds | [41, 52, 63] | [41, 52, 63] |
| Max epochs | 50 | 25 |
| LR | 1e-3 | 3e-4 (head) → 1e-4 (full) |


In [ ]:
SAVE_DIR = "/content/drive/My Drive/Colab Notebooks"


In [ ]:
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")


In [ ]:
results, best = train_multi_seed(
    model_fn     = MobileNetV2_Pretrained,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    seeds        = [41, 52, 63],
    save_dir     = SAVE_DIR,
    name_prefix  = "mobilenetv2_pretrained",
    pretrained   = True,
    # Standard pretrained hyperparameters
    epochs          = 25,
    lr_phase1       = 3e-4,
    lr_phase2       = 1e-4,
    lr_phase3       = 1e-4,    # keep same LR — no third phase needed for V2
    phase2_epoch    = 10,
    phase3_epoch    = 999,     # disable phase 3
    weight_decay    = 1e-4,
    label_smoothing = 0.1,
    patience        = 10,
)


In [ ]:
plot_history(best, title=f"MobileNetV2 Pretrained (seed {best['seed']})")

accs = [r["best_acc"] for r in results]
print(f"\nMobileNetV2 Pretrained")
print(f"  Mean ± Std : {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%")
print(f"  Best       : {best['best_acc']*100:.2f}% @ epoch {best['best_epoch']} (seed {best['seed']})")
print(f"  Checkpoint : {best['save_path']}")


In [ ]:
# ── Quick parameter/size summary ────────────────────────────────────
m = MobileNetV2_Pretrained()
total, _ = count_params(m)
size = model_size_mb(m)
print(f"MobileNetV2 Pretrained  |  Params: {total/1e6:.2f}M  |  Size: {size:.2f}MB")
print(f"Compare to nb 06 (MobileNetV2 Scratch) to isolate initialization effect.")
